In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import os
import cv2
import matplotlib.image as mpimg

In [ ]:
file_path = os.listdir('Cats_Dogs/')

In [ ]:
import re

In [ ]:
mpimg.imread('Cats_Dogs/cat.0.jpg')

In [ ]:
with open('Cats_Dogs/cat.0.xml','r', encoding='utf-8') as file:
    text = file.read()

In [ ]:

X = []
y = []

file_path = os.listdir('Cats_Dogs/')

for i in file_path:
    try:
        if not i.endswith('.jpg'):
            continue

        image_name = i
        xml_name = i.replace('.jpg', '.xml')

        image = mpimg.imread(f'Cats_Dogs/{image_name}')

        with open(f'Cats_Dogs/{xml_name}', 'r', encoding='utf-8') as xml_file:
            coords = re.findall(
                r'<(?:xmin|ymin|xmax|ymax)>(\d+)</(?:xmin|ymin|xmax|ymax)>',
                xml_file.read()
            )

        xmin, ymin, xmax, ymax = coords

        image_x = image.shape[1]
        image_y = image.shape[0]

        xmin = int(xmin)
        ymin = int(ymin)
        xmax = int(xmax)
        ymax = int(ymax)

        xmin = int(xmin * 300 / image_x)
        xmax = int(xmax * 300 / image_x)
        ymin = int(ymin * 300 / image_y)
        ymax = int(ymax * 300 / image_y)

        image = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
        image = cv2.resize(image, (300, 300))
        image = image / 255
        image = image.reshape(300, 300, 1)

        X.append(image)
        y.append([xmin, ymin, xmax, ymax])

    except Exception as e:
        print(f'Ошибка в {i}: {e}')
        continue

X = np.array(X)
y = np.array(y)

print(X.shape)
print(y.shape)


In [ ]:
X.shape

In [ ]:
y.shape

In [ ]:
from keras import Sequential
from keras.layers import Conv2D, MaxPooling2D, Flatten, Dense
from keras.optimizers import Adam

In [ ]:
model = Sequential()
model.add(Conv2D(filters=32, kernel_size=(3, 3), activation='relu', input_shape=(300, 300, 1)))
model.add(Conv2D(filters=64, kernel_size=(3, 3), activation='relu'))
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Conv2D(filters=128, kernel_size=(3, 3), activation='relu'))
model.add(MaxPooling2D(pool_size=(2, 2)))

model.add(Flatten())

model.add(Dense(128, activation='relu'))
model.add(Dense(64, activation='relu'))
model.add(Dense(32, activation='relu'))
model.add(Dense(4, activation='linear'))

In [ ]:

from sklearn.model_selection import train_test_split

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42
)

print("Train:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)
print("Test:", X_test.shape, y_test.shape)


In [ ]:
model.compile(optimizer=Adam(learning_rate=0.001), loss='mae')

In [ ]:
plt.imshow(test_image)

In [ ]:

test_image = X_test[0]

prediction = model.predict(test_image.reshape(1, 300, 300, 1))

xmin, ymin, xmax, ymax = prediction[0].astype(int)

image = (test_image.reshape(300, 300) * 255).astype(np.uint8)
image = cv2.cvtColor(image, cv2.COLOR_GRAY2RGB)

cv2.rectangle(image, (xmin, ymin), (xmax, ymax), (255, 0, 0), 2)

xmin, ymin, xmax, ymax = y_test[0].astype(int)

cv2.rectangle(image, (xmin, ymin), (xmax, ymax), (0, 255, 0), 2)

plt.figure(figsize=(7, 7))
plt.imshow(image)
plt.title("Red = Prediction | Green = Real")
plt.axis('off')
plt.show()
